In [1]:
from pathlib import Path
import pandas as pd, numpy as np

In [2]:
ROOT = Path.cwd().resolve().parent
DATA, DOCS, RESULTS = ROOT/"data", ROOT/"docs", ROOT/"results"
for p in (DOCS, RESULTS): p.mkdir(exist_ok=True)

In [3]:
import shutil
obs = pd.read_csv(DOCS/"observations.csv", dtype=str)
want = {"black_sigatoka": 12, "yellow_sigatoka": 12, "banana_healthy_leaf": 6}
parts = []
for c, n in want.items():
    s = obs[obs.label == c].sample(n, random_state=11).copy()
    s["set"] = ["dev" if k % 2 == 0 else "test" for k in range(n)]
    parts.append(s)
batch = pd.concat(parts).reset_index(drop=True)
batch["flat"] = batch.file.str.replace("/", "__")
batch.to_csv(DOCS/"annotation_batch1.csv", index=False)

out = DATA/"annotation_batch1"; out.mkdir(exist_ok=True)
for r in batch.itertuples():
    shutil.copy(DATA/"raw"/r.file, out/r.flat)
shutil.make_archive(str(DATA/"annotation_batch1"), "zip", out)
print(batch.groupby(["label", "set"]).size())

label                set 
banana_healthy_leaf  dev     3
                     test    3
black_sigatoka       dev     6
                     test    6
yellow_sigatoka      dev     6
                     test    6
dtype: int64
